## Semantic-Based Topic Modeling

In [1]:
from datetime import datetime
date = datetime.now()
formatted_date = date.strftime("%B %d, %Y")
print(formatted_date)

February 24, 2025


In [2]:
# Check GPU memory
!nvidia-smi

Mon Feb 24 04:10:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Check system RAM
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       2.1Gi        55Gi       6.0Mi        25Gi        80Gi
Swap:             0B          0B          0B


#### Setting up the computing environment

In [4]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

# Set up the current working directory within the Google Drive
%cd /content/drive/My\ Drive/Colab\ Notebooks/LLM/sped_biblio/topic_modeling

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/Colab Notebooks/LLM/sped_biblio/topic_modeling


In [5]:
!pip install --upgrade -q pandas==2.2.2 numpy==1.26.4 sentence-transformers bertopic scikit-learn matplotlib umap-learn hdbscan
!pip install -q dill qgrid
# !pip install -U sentence-transformers transformers
# !pip install spacy
# !python -m spacy download en_core_web_md
!pip install -q python-dotenv
!pip install -q openai --upgrade

In [6]:
import re
import warnings
from collections import defaultdict
import pickle
from pickle import UnpicklingError
from pickle import PicklingError
import itertools

# Data Manipulation
import dill
import numpy as np
import pandas as pd
import requests
import qgrid
import requests
import os
import re

# Natural Language Processing
import nltk
nltk.download('wordnet')
import spacy
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

# Generating Topic Labels
import openai
from dotenv import load_dotenv

# Clustering
from hdbscan import HDBSCAN
from umap import UMAP
from scipy.cluster import hierarchy as sch

# Visualization Imports
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.ticker import FuncFormatter
import colorlover as cl
import textwrap

# Network Analysis
import networkx as nx

# Progress Bar
from tqdm import tqdm

# Display HTML
from IPython.display import IFrame

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

def generate_topic_labels(topic_info, topic_model, openai_model="gpt-4o", max_tokens=10, temperature=0.3):

    gen_names = []

    for topic_id in topic_info['Topic']:
        if topic_id == -1:
            gen_names.append("Outlier")
            continue

        keywords = topic_model.get_topic(topic_id)
        if keywords:
            top_keywords = ", ".join([keyword[0] for keyword in keywords[:10]])
            prompt = f"""
You are a highly skilled data scientist specializing in generating concise and descriptive topic labels based on provided top terms for each topic.
Each topic consists of a list of terms ordered from most to least significant.

Your objective is to create precise labels that capture the essence of each topic by following these guidelines:

1. Use Person-First Language:
   - Prioritize respectful and inclusive language.
   - Avoid terms that may be considered offensive or stigmatizing.
   - For example, use "students with learning disabilities" instead of "disabled students".

2. Analyze the significance of the top terms:
   - Focus primarily on the most significant terms.
   - Include additional terms if they add essential context.

3. Synthesize the Topic Label:
   - Ensure clarity and conciseness (aim for 5-7 words).
   - Reflect the collective meaning of the most influential terms.
   - Use descriptive yet precise phrasing.

4. Maintain consistency:
   - Capitalize the first word using title case.
   - Use uniform formatting and avoid ambiguity.

Example
----------
Top 10 Keywords in [Representation]:
virtual manipulatives, manipulatives, mathematical, app, solving, learning disability, algebra, area, tool, concrete manipulatives

Generated Topic Label in [GenName]:
Visual-Based Technology for Mathematical Problem Solving

Top 10 Keywords: {top_keywords}
Generated Topic Label in [GenName]:
"""
            response = openai.chat.completions.create(
                model=openai_model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=temperature
            )
            gen_name = response.choices[0].message.content.strip()
            gen_name = re.sub(r'Generated Topic Label in \[GenName\]:', '', gen_name, flags=re.IGNORECASE).strip()
            gen_names.append(gen_name)
        else:
            gen_names.append("No Keywords")

    topic_info['GenName'] = gen_names
    return topic_info

#### Combine text columns

In [8]:
all_data_file = f"files/all_data.xlsx"
all_data = pd.read_excel(all_data_file, na_filter=False)

df = all_data[all_data['filtered'] == 'Yes'].reset_index(drop=True)
df['Year'] = df['PY'].astype(int)
df['Decade'] = (df['Year'] // 10) * 10
df['CR'] = df['CR'].astype(str)

#### Cluster documents

In [9]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

In [10]:
publication_embeddings = sentence_model.encode(df['combined_text'].tolist(), show_progress_bar=True)
publication_embeddings_df = pd.DataFrame(publication_embeddings)
publication_embeddings_df['UT'] = df['UT'].tolist()

Batches:   0%|          | 0/108 [00:00<?, ?it/s]

In [11]:
umap_model = UMAP(
    n_neighbors=5,
    n_components=3,
    min_dist=0.01,
    metric='cosine',
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(publication_embeddings_df.drop(columns='UT'))
reduced_embeddings_df = pd.DataFrame(reduced_embeddings, columns=["x", "y", "z"])
reduced_embeddings_df['UT'] = publication_embeddings_df['UT'].tolist()

hdbscan_model = HDBSCAN(
    min_cluster_size=35,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

hdbscan_model.fit(reduced_embeddings)
labels = hdbscan_model.labels_

#### Topic modeling

In [12]:
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(df['combined_text'])

In [13]:
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    tokens = text.split()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmatized_tokens)

def preprocess_texts(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = text.split()
    return ' '.join(tokens)

df['lemmatized_text'] = df['combined_text'].apply(lemmatize_text)

df['preprocessed_text'] = df['lemmatized_text'].apply(preprocess_texts)

vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,
  calculate_probabilities=True,
  verbose=True
)

In [14]:
topics, probs = topic_model.fit_transform(df['preprocessed_text'])
df['Topic'] = topics
for i in range(probs.shape[1]):
    df[f'Prob_Topic_{i}'] = probs[:, i]

topic_model.update_topics(df['preprocessed_text'], vectorizer_model=vectorizer_model)

topic_info = topic_model.get_topic_info()

data = []
for topic in topic_info['Topic']:
    if topic == -1:
        continue
    words_with_scores = topic_model.get_topic(topic)
    for word, score in words_with_scores:
        data.append({"Topic": topic, "Word": word, "Score": score})

topic_model_words = pd.DataFrame(data)

topic_info_words_combined = topic_info.merge(topic_model_words, how="left", on="Topic") \
                                                      .sort_values(by=['Topic', 'Score'], ascending=[True, False])

2025-02-24 04:12:21,780 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2025-02-24 04:12:24,431 - BERTopic - Embedding - Completed ✓
2025-02-24 04:12:24,431 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-24 04:12:39,986 - BERTopic - Dimensionality - Completed ✓
2025-02-24 04:12:39,987 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-24 04:12:40,243 - BERTopic - Cluster - Completed ✓
2025-02-24 04:12:40,247 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-24 04:12:46,046 - BERTopic - Representation - Completed ✓


In [15]:
topic_info_gen = generate_topic_labels(topic_info, topic_model)

topic_model.set_topic_labels(topic_info_gen['GenName'].tolist())

In [16]:
df_filtered = df[df['Topic'] == -1].reset_index(drop=True)
docs_filtered = df_filtered['preprocessed_text'].tolist()

umap_model_outliers = UMAP(
    n_neighbors=10,
    n_components=3,
    min_dist=0.05,
    metric='cosine',
    random_state=42
)

hdbscan_model_outliers = HDBSCAN(
    min_cluster_size=25,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

reduced_embeddings_outliers = umap_model_outliers.fit_transform(publication_embeddings_df.drop(columns='UT'))
reduced_embeddings_outliers_df = pd.DataFrame(reduced_embeddings_outliers, columns=["x", "y", "z"])
reduced_embeddings_outliers_df['UT'] = publication_embeddings_df['UT'].tolist()

hdbscan_model_outliers.fit(reduced_embeddings_outliers)
labels = hdbscan_model_outliers.labels_

topic_model_outliers = BERTopic(
    embedding_model=sentence_model,
    umap_model=umap_model_outliers,
    hdbscan_model=hdbscan_model_outliers,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True
)

new_topics, new_probs = topic_model_outliers.fit_transform(docs_filtered)
df_filtered['Topic'] = new_topics
for i in range(new_probs.shape[1]):
    df_filtered[f'Prob_New_Topic_{i}'] = new_probs[:, i]

topic_model_outliers.update_topics(docs_filtered, vectorizer_model=vectorizer_model)

topic_info_filtered = topic_model_outliers.get_topic_info()

data = []
for topic in topic_info_filtered['Topic']:
    if topic == -1:
        continue
    words_with_scores = topic_model_outliers.get_topic(topic)
    for word, score in words_with_scores:
        data.append({"Topic": topic, "Word": word, "Score": score})

topic_model_outliers_words = pd.DataFrame(data)

topic_info_filtered_words_combined = topic_info_filtered.merge(topic_model_outliers_words, how="left", on="Topic")

2025-02-24 04:13:17,899 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

2025-02-24 04:13:18,534 - BERTopic - Embedding - Completed ✓
2025-02-24 04:13:18,535 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-24 04:13:20,196 - BERTopic - Dimensionality - Completed ✓
2025-02-24 04:13:20,197 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-24 04:13:20,242 - BERTopic - Cluster - Completed ✓
2025-02-24 04:13:20,244 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-24 04:13:21,610 - BERTopic - Representation - Completed ✓


In [17]:
topic_info_filtered_gen = generate_topic_labels(topic_info_filtered, topic_model_outliers)

topic_model_outliers.set_topic_labels(topic_info_filtered_gen['GenName'].tolist())

In [18]:
df_filtered_updated = df_filtered[df_filtered['Topic'] == -1].reset_index(drop=True)
docs_filtered_updated = df_filtered_updated['preprocessed_text'].tolist()

umap_model_outliers_updated = UMAP(
    n_neighbors=10,
    n_components=3,
    min_dist=0.05,
    metric='cosine',
    random_state=42
)

hdbscan_model_outliers_updated = HDBSCAN(
    min_cluster_size=10,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

reduced_embeddings_outliers_updated = umap_model_outliers_updated.fit_transform(publication_embeddings_df.drop(columns='UT'))
reduced_embeddings_outliers_updated_df = pd.DataFrame(reduced_embeddings_outliers_updated, columns=["x", "y", "z"])
reduced_embeddings_outliers_updated_df['UT'] = publication_embeddings_df['UT'].tolist()

hdbscan_model_outliers_updated.fit(reduced_embeddings_outliers_updated)
labels = hdbscan_model_outliers_updated.labels_

topic_model_outliers_updated = BERTopic(
    embedding_model=sentence_model,
    umap_model=umap_model_outliers_updated,
    hdbscan_model=hdbscan_model_outliers_updated,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True
)

new_topics_updated, new_probs_updated = topic_model_outliers_updated.fit_transform(docs_filtered_updated)
df_filtered_updated['Topic'] = new_topics_updated
for i in range(new_probs_updated.shape[1]):
    df_filtered_updated[f'Prob_New_Topic_Updated_{i}'] = new_probs_updated[:, i]

topic_model_outliers_updated.update_topics(docs_filtered_updated, vectorizer_model=vectorizer_model)

topic_info_filtered_updated = topic_model_outliers_updated.get_topic_info()

data = []
for topic in topic_info_filtered_updated['Topic']:
    if topic == -1:
        continue
    words_with_scores = topic_model_outliers_updated.get_topic(topic)
    for word, score in words_with_scores:
        data.append({"Topic": topic, "Word": word, "Score": score})

topic_model_outliers_updated_words = pd.DataFrame(data)

topic_info_filtered_updated_words_combined = topic_info_filtered_updated.merge(topic_model_outliers_updated_words, how="left", on="Topic")

2025-02-24 04:13:46,283 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

2025-02-24 04:13:46,473 - BERTopic - Embedding - Completed ✓
2025-02-24 04:13:46,473 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-24 04:13:46,812 - BERTopic - Dimensionality - Completed ✓
2025-02-24 04:13:46,813 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-24 04:13:46,827 - BERTopic - Cluster - Completed ✓
2025-02-24 04:13:46,829 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-24 04:13:47,287 - BERTopic - Representation - Completed ✓


In [19]:
topic_info_filtered_updated_gen = generate_topic_labels(topic_info_filtered_updated, topic_model_outliers_updated)

topic_model_outliers_updated.set_topic_labels(topic_info_filtered_updated_gen['GenName'].tolist())

In [20]:
topic_info_no_rep_doc = topic_info[topic_info['Topic'] != -1].drop(columns=['Representative_Docs'])
df_topic_info = df[df['Topic'] !=-1].merge(topic_info_no_rep_doc, how="left", on="Topic")
df_topic_info = reduced_embeddings_df.merge(df_topic_info, how="inner", on="UT")

In [21]:
len(df_topic_info)

2688

In [22]:
topic_info_no_rep_doc_filtered = topic_info_filtered[topic_info_filtered['Topic'] != -1].drop(columns=['Representative_Docs'])
df_topic_info_filtered = df_filtered[df_filtered['Topic'] !=-1].merge(topic_info_no_rep_doc_filtered, how="left", on="Topic")
df_topic_info_filtered = reduced_embeddings_outliers_df.merge(df_topic_info_filtered, how="inner", on="UT")

In [23]:
len(df_topic_info_filtered)

535

In [24]:
topic_info_no_rep_doc_filtered_updated = topic_info_filtered_updated[topic_info_filtered_updated['Topic'] != -1].drop(columns=['Representative_Docs'])
df_topic_info_filtered_updated = df_filtered_updated[df_filtered_updated['Topic'] !=-1].merge(topic_info_no_rep_doc_filtered_updated, how="left", on="Topic")
df_topic_info_filtered_updated = reduced_embeddings_outliers_updated_df.merge(df_topic_info_filtered_updated, how="inner", on="UT")

In [25]:
len(df_topic_info_filtered_updated)

206

In [26]:
topic_info_combined_concat = pd.concat([topic_info_no_rep_doc, topic_info_no_rep_doc_filtered, topic_info_no_rep_doc_filtered_updated], ignore_index=True)
topic_info_combined_concat = topic_info_combined_concat[topic_info_combined_concat['Topic'] != -1].drop(columns=['Topic'])

In [38]:
topic_info_combined_concat_human_loop = pd.read_csv("files/topic_info_combined_concat_human_loop.csv")

In [39]:
topic_info_combined_concat = pd.merge(topic_info_combined_concat, topic_info_combined_concat_human_loop[['Name', 'Topic', 'CustomLabel']], how="left", on="Name")

first_column = 'Topic'
original_columns = topic_info_combined_concat.columns.tolist()
if first_column in original_columns:
    original_columns.remove(first_column)
    new_columns = [first_column] + original_columns
else:
    new_columns = original_columns

topic_info_combined_concat = topic_info_combined_concat.reindex(columns=new_columns)

In [40]:
topic_info_words_combined_concat = pd.concat([topic_info_words_combined, topic_info_filtered_words_combined, topic_info_filtered_updated_words_combined], ignore_index=True)
topic_info_words_combined_concat = topic_info_words_combined_concat[topic_info_words_combined_concat['Topic'] != -1].drop(columns=['Topic'])

topic_info_words_combined_concat = pd.merge(topic_info_words_combined_concat, topic_info_combined_concat_human_loop[['Name', 'Topic', 'GenName', 'CustomLabel']], how="left", on="Name")

first_column = 'Topic'
original_columns = topic_info_words_combined_concat.columns.tolist()
if first_column in original_columns:
    original_columns.remove(first_column)
    new_columns = [first_column] + original_columns
else:
    new_columns = original_columns

topic_info_words_combined_concat = topic_info_words_combined_concat.reindex(columns=new_columns)

In [41]:
df_combined_concat = pd.concat([df_topic_info, df_topic_info_filtered, df_topic_info_filtered_updated], ignore_index=True)
df_combined_concat = df_combined_concat[df_combined_concat['Topic'] != -1].drop(columns=['Topic'])

df_combined_concat = pd.merge(df_combined_concat, topic_info_combined_concat_human_loop[['Name', 'Topic', 'CustomLabel']], how="left", on="Name")

key_cols = ['Topic', 'Count', 'Name', 'Representation', 'GenName', 'CustomLabel']
all_cols = df_combined_concat.columns.tolist()
remaining_cols = [col for col in all_cols if col not in key_cols]
index_pos = remaining_cols.index('Prob_Topic_0')
final_order = remaining_cols[:index_pos] + key_cols + remaining_cols[index_pos:]
df_combined_concat = df_combined_concat[final_order]

In [ ]:
init_df = df.copy()
init_topic_info = topic_info.copy()

files = {
    "files/topic_model.pkl": topic_model,
    "files/topic_model_outliers.pkl": topic_model_outliers,
    "files/topic_model_outliers_updated.pkl": topic_model_outliers_updated,
    "files/publication_embeddings.pkl": publication_embeddings,
    "files/reduced_embeddings.pkl": reduced_embeddings,
    "files/topic_info.pkl": topic_info,
    "files/topic_info_filtered.pkl": topic_info_filtered,
    "files/topic_info_filtered_updated.pkl": topic_info_filtered_updated,
    "files/topic_info_combined_concat.pkl": topic_info_combined_concat,
    "files/topic_info_words_combined_concat.pkl": topic_info_words_combined_concat,
    "files/df_topic_info.pkl": df_topic_info,
    "files/df_topic_info_filtered.pkl": df_topic_info_filtered,
    "files/df_topic_info_filtered_updated.pkl": df_topic_info_filtered_updated,
    "files/df_combined_concat.pkl": df_combined_concat
}

topic_info.to_csv("files/topic_info.csv", index=False)
topic_info_filtered.to_csv("files/topic_info_filtered.csv", index=False)
topic_info_filtered_updated.to_csv("files/topic_info_filtered_updated.csv", index=False)
topic_info_combined_concat.to_csv("files/topic_info_combined_concat.csv", index=False)
topic_info_words_combined_concat.to_csv("files/topic_info_words_combined_concat.csv", index=False)
df_combined_concat.to_csv("files/df_combined_concat.csv", index=False)

for filename, data in files.items():
    try:
        print(f"Attempting to save {filename}...")
        if data is not None:
          print(f"  Data type: {type(data)}")
          if hasattr(data, '__len__'):
              print(f"  Data length: {len(data)}")
          with open(filename, "wb") as f:
              pickle.dump(data, f)
          print(f"Successfully saved {filename}.")
        else:
            print(f"Error: Data for {filename} is None, skipping save")
    except PicklingError as e:
       print(f"Error pickling {filename}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while writing {filename}: {e}")

In [32]:
# loaded_data = {}

# for filename, data in files.items():
#    if os.path.exists(filename):
#      file_size = os.path.getsize(filename)
#      if file_size > 0:
#          try:
#              with open(filename, "rb") as f:
#                  loaded_data[filename] = pickle.load(f)
#              print(f"Successfully loaded {filename}")
#          except EOFError as e:
#              print(f"Error: Could not load {filename}. It might be empty or corrupted: {e}")
#              loaded_data[filename] = None
#          except Exception as e:
#            print(f"An unexpected error occurred while loading {filename}: {e}")
#            loaded_data[filename] = None
#      else:
#          print(f"Error: File is empty: {filename}")
#          loaded_data[filename] = None
#    else:
#        print(f"Error: File not found: {filename}")
#        loaded_data[filename] = None

# topic_model = loaded_data.get("files/topic_model.pkl")
# topic_model_outliers = loaded_data.get("files/topic_model_outliers.pkl")
# topic_model_outliers_updated = loaded_data.get("files/topic_model_outliers_updated.pkl")
# publication_embeddings = loaded_data.get("files/publication_embeddings.pkl")
# reduced_embeddings = loaded_data.get("files/reduced_embeddings.pkl")
# topic_info = loaded_data.get("files/topic_info.pkl")
# topic_info_filtered = loaded_data.get("files/topic_info_filtered.pkl")
# topic_info_filtered_updated = loaded_data.get("files/topic_info_filtered_updated.pkl")
# topic_info_combined_concat = loaded_data.get("files/topic_info_combined_concat.pkl")
# topic_info_words_combined_concat = loaded_data.get("files/topic_info_words_combined_concat.pkl")
# df_topic_info = loaded_data.get("files/df_topic_info.pkl")
# df_topic_info_filtered = loaded_data.get("files/df_topic_info_filtered.pkl")
# df_topic_info_filtered_updated = loaded_data.get("files/df_topic_info_filtered_updated.pkl")
# df_combined_concat = loaded_data.get("files/df_combined_concat.pkl")

# topic_info = pd.read_csv("files/topic_info.csv")
# topic_info_filtered = pd.read_csv("files/topic_info_filtered.csv")
# topic_info_filtered_updated = pd.read_csv("files/topic_info_filtered_updated.csv")
# topic_info_combined_concat = pd.read_csv("files/topic_info_combined_concat.csv")
# topic_info_words_combined_concat = pd.read_csv("files/topic_info_words_combined_concat.csv")
# df_combined_concat = pd.read_csv("files/df_combined_concat.csv")

In [ ]:
df = topic_info_words_combined_concat.copy()

replacements = {
    "augmentative alternative communication": "AAC",
    "aac": "AAC",
    "aba": "ABA",
    "applied behavior analysis": "ABA",
    "gbg": "GBG",
    "good behavior game": "GBG"
}

def apply_replacements(label):
    """Performs case-insensitive replacement for each key in replacements."""
    label = label.strip()
    for old, new in replacements.items():
        pattern = re.compile(re.escape(old), flags=re.IGNORECASE)
        label = pattern.sub(new, label)
    return label

custom_labels = {t: df.loc[df['Topic'] == t, 'CustomLabel'].iloc[0] for t in sorted(df['Topic'].unique())}

topics = sorted(df['Topic'].unique())
num_charts = len(topics)

custom_labels_transformed = {t: apply_replacements(label) for t, label in custom_labels.items()}
custom_labels_values = list(custom_labels_transformed.values())

subplot_titles = [
    " ".join(t.split()[:len(t.split()) // 2]) + "<br>" + " ".join(t.split()[len(t.split()) // 2:])
    if len(t) > 10 else t
    for t in itertools.islice(custom_labels_values, num_charts)
]

num_columns = 4
num_rows = (num_charts + num_columns - 1) // num_columns

color_map = [
  '#AB63FA', '#637939', '#EF553B', '#d62728', '#FF97FF',
  '#17becf', '#1f77b4', '#e377c2', '#9cf4f7', '#BCBD22',
  '#7b4173', '#a799e8', '#8c564b', '#00CC96', '#393b79',
  '#5254a3', '#FECB52', '#2ca02c', '#FFA15A', '#9467bd',
  '#9cd4f7', '#fb9a99', '#8c6d31', '#B6E880', '#636EFA',
  '#843c39', '#19D3F3', '#bcbd22', '#7CF3A0', '#FF6692',
  '#7f7f7f', '#ff7f0e'
]

topic_color_map = {t: color_map[i % len(color_map)] for i, t in enumerate(topics)}

fig = make_subplots(
    rows=num_rows,
    cols=num_columns,
    vertical_spacing=0.05,
    subplot_titles=subplot_titles
)

for idx, t in enumerate(topics):
    df_topic = df[df['Topic'] == t].copy()
    df_topic['Word'] = df_topic['Word'].apply(apply_replacements)
    aggregated = df_topic.groupby('Word', as_index=False)['Score'].sum()
    aggregated = aggregated.sort_values(by='Score', ascending=False)
    aggregated = aggregated.head(5)

    row = (idx // num_columns) + 1
    col = (idx % num_columns) + 1
    if aggregated.empty:
        trace = go.Bar(
            x=[], y=[], orientation='h',
            marker=dict(color=topic_color_map[t]),
            name=f"Topic {t}"
        )
    else:
        trace = go.Bar(
            x=aggregated['Score'],
            y=aggregated['Word'],
            orientation='h',
            marker=dict(color=topic_color_map[t]),
            name=f"Topic {t}"
        )
    fig.add_trace(trace, row=row, col=col)

# Update the layout and axes.
fig.update_layout(
    title_text="<b>Topic Word Scores</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=120, b=120, l=80, r=80),
    width=1900,
    height=1500,
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

for r in range(1, num_rows + 1):
    for c in range(1, num_columns + 1):
        fig.update_xaxes(
            tickfont=dict(size=14),
            automargin=True,
            row=r, col=c
        )
        fig.update_yaxes(
            ticklabelstandoff=10,
            tickfont=dict(size=14),
            automargin=True,
            autorange="reversed",
            row=r, col=c
        )

for annotation in fig.layout.annotations:
    annotation.font.size = 14

fig.write_html("results/topic_word_barchart.html")
# fig.show()

In [49]:
IFrame(src='results/topic_word_barchart.html', width=2000, height=1200)

In [ ]:
plot_data = df_combined_concat.copy()
plot_data['Topic'] = pd.to_numeric(plot_data['Topic'], errors='coerce')
plot_data['x'] = pd.to_numeric(plot_data['x'], errors='coerce')
plot_data['y'] = pd.to_numeric(plot_data['y'], errors='coerce')
plot_data = plot_data.dropna(subset=['Topic', 'x', 'y'])
plot_data['Topic'] = plot_data['Topic'].astype(int)

ordered_topics = sorted(plot_data['Topic'].unique())
ordered_legend = ['Topic ' + str(t) for t in ordered_topics]

plot_data['legend_topic'] = 'Topic ' + plot_data['Topic'].astype(str)
plot_data['legend_topic'] = plot_data['legend_topic'].astype("category")
plot_data['legend_topic'] = plot_data['legend_topic'].cat.set_categories(ordered_legend, ordered=True)

plot_data['hover_text'] = (
    "Title: " +
    plot_data['TI'].apply(lambda text: "<br>".join(textwrap.wrap(text, width=50))) +
    "<br><br>Topic " + plot_data['Topic'].astype(str) + ": " + plot_data['CustomLabel']
)

fig_cluster = px.scatter(
    plot_data,
    x='x',
    y='y',
    color='legend_topic',
    color_discrete_map={'Topic ' + str(t): topic_color_map[t] for t in ordered_topics},
    labels={'x': 'X', 'y': 'Y'},
    custom_data=['hover_text'],
    category_orders={'legend_topic': ordered_legend}
)


fig_cluster.update_traces(
    marker=dict(size=7, opacity=0.9, line=dict(width=0.6, color="black")),
    selector=dict(mode='markers'),
    hovertemplate='<b>%{customdata[0]}</b><extra></extra>'
)

fig_cluster.update_layout(
    title="<b>Documents and Topics</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=80, b=80, l=80, r=80),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=800,
    height=600,
    showlegend=True,
    legend_title_text="Topic"
)

fig_cluster.write_html("results/docs_topics.html")
# fig_cluster.show()

In [57]:
IFrame(src='results/docs_topics.html', width=1000, height=600)

In [61]:
from nbconvert import HTMLExporter
import nbformat

notebook_path = 'index.ipynb'
html_exporter = HTMLExporter()

with open(notebook_path, 'r', encoding='utf-8') as nb_file:
    notebook_content = nb_file.read()
    notebook = nbformat.reads(notebook_content, as_version=4)

if 'widgets' in notebook.metadata and 'application/vnd.jupyter.widget-state+json' in notebook.metadata['widgets']:
    if 'state' not in notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']:
        notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']['state'] = {}

html_output, _ = html_exporter.from_notebook_node(notebook)

with open('index.html', 'w', encoding='utf-8') as html_file:
    html_file.write(html_output)